# What is LangGraph?

**LangGraph** is an orchestration framework for building **intelligent, stateful, multi-step LLM workflows**.

### Key Features

- **Stateful** → Maintains state/data throughout the workflow
- **Parallel execution** → Run multiple tasks simultaneously
- **Loops** → Repeat steps when required
- **Branching** → Choose different paths based on conditions
- **Memory** → Maintain information across interactions
- **Resumability** → Pause and resume workflow execution
- **Production-ready** → Suitable for robust agentic AI applications

### Core Concept

LangGraph represents workflow logic as a **graph**:

- **Nodes** → Tasks/functions
- **Edges** → Routing between tasks
- **State** → Data shared across the workflow

```text
        Node A
          ↓
        Node B
       ↙     ↘
   Node C    Node D
       ↘     ↙
        Node E

# Prompt Chaining vs Routing

**Prompt Chaining** → Executes prompts **sequentially**, where the output of one step becomes the input of the next.

`Input → Prompt 1 → Prompt 2 → Prompt 3 → Output`

**Routing** → A **router/condition** decides which prompt or workflow should handle the input.

`Input → Router → Prompt A / Prompt B / Prompt C → Output`

### Key Difference

- 🔗 **Chaining** = sequential flow
- 🔀 **Routing** = conditional flow
- **Chaining:** `A → B → C`
- **Routing:** `A → B OR C`

**Example:**

`Question → Summarize → Extract → Generate` = **Chaining**

`Question → Router → Billing OR Technical` = **Routing**

# Parallelization

**Parallelization** → Executes **multiple independent tasks simultaneously** instead of one after another.

<table>
<tr>
<td rowspan="3" align="center"><b>Input</b></td>
<td rowspan="3" align="center">→</td>
<td align="center"><b>Task A</b></td>
<td rowspan="3" align="center">→</td>
<td rowspan="3" align="center"><b>Aggregator</b></td>
<td rowspan="3" align="center">→</td>
<td rowspan="3" align="center"><b>Output</b></td>
</tr>
<tr>
<td align="center"><b>Task B</b></td>
</tr>
<tr>
<td align="center"><b>Task C</b></td>
</tr>
</table>


### Example

Aggregator = collect/combine the results from those parallel tasks.

For a user query, an AI system can simultaneously:

- 🔍 Search the web
- 📄 Retrieve documents
- 🧠 Analyze the query

Then combine the results.

### Key Difference

- **Sequential:** `A → B → C`
- **Parallel:** `A + B + C → Output`

**Use when:** Tasks are **independent** and don't need the output of another task to start.

> **Easy to remember:**  
> ⚡ **Parallelization = Independent tasks run at the same time to reduce overall execution time.**

# Orchestrator-Workers

**Orchestrator-Workers** → An **orchestrator** dynamically breaks a task into smaller subtasks and assigns them to **workers**. The workers execute the subtasks, and their results are combined into a final output.

### Workflow

<table>
<tr>
<td rowspan="3" align="center"><b>Input</b></td>
<td rowspan="3" align="center">→</td>
<td rowspan="3" align="center"><b>Orchestrator</b></td>
<td align="center">→ <b>Worker A</b></td>
<td rowspan="3" align="center">→</td>
<td rowspan="3" align="center"><b>Aggregator</b></td>
<td rowspan="3" align="center">→</td>
<td rowspan="3" align="center"><b>Output</b></td>
</tr>
<tr>
<td align="center">→ <b>Worker B</b></td>
</tr>
<tr>
<td align="center">→ <b>Worker C</b></td>
</tr>
</table>

### Components

- **Orchestrator** → Analyzes the task and decides how to split it.
- **Workers** → Execute the assigned subtasks independently.
- **Aggregator** → Combines worker results into the final result.

### Example

For **"Research AI trends and create a report"**:

- Orchestrator → Splits the task
- Worker A → Research AI trends
- Worker B → Research market trends
- Worker C → Analyze competitors
- Aggregator → Combines all results
- Output → Final report

### Key Difference

**Parallelization:** Tasks are predefined and run in parallel.

**Orchestrator-Workers:** Tasks are **dynamically created/assigned by the orchestrator** based on the input.

> 🧠 **Easy to remember:**  
> **Orchestrator = Plans & assigns**  
> **Workers = Execute**  
> **Aggregator = Combines**

# Evaluator-Optimizer

**Evaluator-Optimizer** → A workflow where one component **generates/improves an output**, while another component **evaluates it** and provides feedback. The process repeats until the output meets the required quality.

### Workflow

<table>
<tr>
<td align="center"><b>Input</b></td>
<td align="center">→</td>
<td align="center"><b>Optimizer</b></td>
<td align="center">→</td>
<td align="center"><b>Evaluator</b></td>
<td align="center">→</td>
<td align="center"><b>Final Output</b></td>
</tr>
</table>

### Iterative Feedback Loop

**Optimizer → Evaluator → Feedback → Optimizer → Evaluator → ...**

**If evaluation fails:**

`Optimizer → Evaluator → Feedback → Optimizer → ...`

### Components

- **Optimizer** → Generates or improves the response.
- **Evaluator** → Checks the response against quality criteria.
- **Feedback** → Tells the optimizer what needs improvement.
- **Final Output** → Returned when the evaluation passes.

### Example

**Generate code → Evaluate code → Find issues → Improve code → Evaluate again**

### Key Idea

> 🔄 **Optimizer creates/improves → Evaluator checks → Feedback → Repeat**

**Use when:** The output requires **iterative improvement** based on evaluation or feedback.

# State Component

**State** → A shared data structure that stores the **current information and context** of a LangGraph workflow.

**State** is mutable

- **Nodes read** data from the State.
- **Nodes update** the State.
- The updated State is passed to the next nodes.

### Workflow

`Input → State → Node → Updated State → Next Node`

### Defining State

LangGraph can use both **TypedDict** and **Pydantic** to define State.

**TypedDict** → Lightweight; mainly provides type hints, **no runtime validation**.

**Pydantic** → Provides type hints **+ runtime data validation**.

```python
# TypedDict
class State(TypedDict):
    user_query: str
    result: str
# Pydantic
class State(BaseModel):
    user_query: str
    result: str


# Reducers

**Reducer** → Defines how a new update is applied to an existing State value.

A reducer can define whether the new data should:

- **Replace** the existing value
- **Merge** with the existing value
- **Add/Append** to the existing value

### Example

```text
Old State + New Update
        ↓
      Reducer
        ↓
 ┌──────┼──────┐
Replace Merge  Add

# LangGraph Execution Model

### 1. Graph Definition
Define:
- **State Schema** → Structure of the workflow state
- **Nodes** → Functions/tasks
- **Edges** → Connections and routing between nodes

### 2. Compilation

Call **`.compile()`** on the `StateGraph`.

- Validates the graph structure
- Prepares the graph for execution

### 3. Invocation

Run the graph using:

`graph.invoke(initial_state)`

The initial state is passed to the entry node(s).

### 4. Super-Steps

LangGraph executes the graph in **rounds called super-steps**.

Each super-step activates the nodes that are ready to execute.

### 5. Message Passing & Node Activation

- State/data is passed between nodes through **edges**.
- Nodes receiving updates become **active** for the next super-step.
- Multiple independent nodes can execute in the same super-step.

### 6. Halting Condition

Execution stops when:

- **No nodes are active**, and
- **No messages/updates are in transit**

> 🧠 **Execution Flow:**  
> **Define Graph → Compile → Invoke → Super-Steps → Node Execution & State Updates → Repeat → Halt**